In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import RandomOverSampler
import gc, os, random, warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_seed(42)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
df = pd.read_csv(r"C:\Users\nuzha\IDD_project\IDD_DS\NF-ToN-IoT.csv")  
df = df.fillna(0)
df.replace([np.inf, -np.inf], 0, inplace=True)

df['IN_BYTE_PER_PKT']    = df['IN_BYTES']  / (df['IN_PKTS']  + 1e-6)
df['OUT_BYTE_PER_PKT']   = df['OUT_BYTES'] / (df['OUT_PKTS'] + 1e-6)
df['TOTAL_BYTES']        = df['IN_BYTES']  + df['OUT_BYTES']
df['TOTAL_PKTS']         = df['IN_PKTS']   + df['OUT_PKTS']
df['FLOW_DUR_PER_PKT']   = df['FLOW_DURATION_MILLISECONDS'] / (df['TOTAL_PKTS'] + 1e-6)
df['IN_OUT_BYTES_RATIO'] = df['IN_BYTES'] / (df['OUT_BYTES'] + 1e-6)
df['IN_OUT_PKTS_RATIO']  = df['IN_PKTS']  / (df['OUT_PKTS']  + 1e-6)
df['TCP_SYN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x02 else 0)
df['TCP_ACK'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x10 else 0)
df['TCP_FIN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x01 else 0)

le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])
feature_cols = [c for c in df.columns if c not in
                ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Label', 'Attack']]

print("Shape:", df.shape)
print("Features:", len(feature_cols))
print(df['Attack'].value_counts())

Shape: (1379274, 24)
Features: 20
Attack
injection     468539
ddos          326345
Benign        270279
password      156299
xss            99944
scanning       21467
dos            17717
backdoor       17247
mitm            1295
ransomware       142
Name: count, dtype: int64


In [3]:
WINDOW, STRIDE = 10, 5
zero_day_attacks = ['scanning', 'dos', 'backdoor', 'ransomware']

def create_sequences(data, feature_cols, window_size=10, stride=5):
    sequences, labels, attacks = [], [], []
    feats = data[feature_cols].values.astype(np.float32)
    labs  = data['Label'].values
    atks  = data['Attack'].values
    for i in range(0, len(feats) - window_size + 1, stride):
        sequences.append(feats[i:i+window_size])
        labels.append(labs[i + window_size - 1])
        attacks.append(atks[i + window_size - 1])
    return (np.array(sequences, dtype=np.float32),
            np.array(labels),
            np.array(attacks))

class FB_HESN_Layer(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3,
                 omega_min=0.3, omega_max=1.0, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
        self.omega_min = omega_min
        self.omega_max = omega_max

    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units),
                                    initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1, 1, (self.units, self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units, self.units),
                                 initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        omega = np.random.uniform(self.omega_min, self.omega_max, self.units).astype(np.float32)
        self.omega = self.add_weight(shape=(self.units,),
                                     initializer=tf.constant_initializer(omega), trainable=False)
        super().build(input_shape)

    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:, t, :], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t + 1, (1 - self.leak_rate) * state + self.leak_rate * tf.sin(self.omega * pre)
        _, final = tf.while_loop(lambda t, s: t < tf.shape(inputs)[1], body, (0, state))
        return final

def build_primary_model(n_features):
    inputs = Input(shape=(10, n_features))
    x = FB_HESN_Layer(units=64, spectral_radius=0.9, leak_rate=0.3,
                      omega_min=0.3, omega_max=1.0)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(2, activation='softmax')(x)
    return Model(inputs, outputs)

def scale_and_oversample(X_train, y_train, X_test, y_test, X_val=None, y_val=None):
    n_features = X_train.shape[-1]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
    X_test  = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)
    if X_val is not None:
        X_val = scaler.transform(X_val.reshape(-1, n_features)).reshape(X_val.shape)

    n_s, t, f = X_train.shape
    X_train_2d, y_train = RandomOverSampler(random_state=42).fit_resample(
        X_train.reshape(n_s, -1), y_train)
    X_train = X_train_2d.reshape(-1, t, f)

    if X_val is not None:
        return X_train, y_train, X_val, y_val, X_test, y_test
    return X_train, y_train, X_test, y_test

def train_and_eval(X_train, y_train, X_test, y_test, name, X_val=None, y_val=None):
    set_seed(42)
    tf.keras.backend.clear_session()
    model = build_primary_model(X_train.shape[-1])
    model.compile(optimizer=Adam(0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    callbacks = [EarlyStopping(monitor='val_loss' if X_val is not None else 'loss',
                               patience=5, restore_best_weights=True, verbose=0)]
    fit_kwargs = dict(epochs=40, batch_size=256, callbacks=callbacks, verbose=1)
    if X_val is not None:
        model.fit(X_train, y_train, validation_data=(X_val, y_val), **fit_kwargs)
    else:
        model.fit(X_train, y_train, **fit_kwargs)

    y_prob = model.predict(X_test, batch_size=256, verbose=0)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    print("\n" + "="*70)
    print(name)
    print("="*70)
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack'], digits=4))
    macro = f1_score(y_test, y_pred, average='macro')
    print("Macro F1:", macro)
    print("Benign support:", int((y_test == 0).sum()))
    return macro

In [4]:
# ============================================================
# CONTROL — chronological split (hosts may overlap)
# Test sequences built PER PARTITION (match primary protocol)
# ============================================================
print("="*70)
print("CONTROL: chronological split (hosts may overlap)")
print("="*70)

df_tmp = df.reset_index(drop=True)
zero_day_df = df_tmp[df_tmp['Attack'].str.lower().isin(zero_day_attacks)].copy()
normal_df   = df_tmp[~df_tmp['Attack'].str.lower().isin(zero_day_attacks)].copy()

# Same 72 / 8 / 20 style as primary
cutoff1 = int(0.80 * len(normal_df))
dev_df      = normal_df.iloc[:cutoff1].copy()
test_normal = normal_df.iloc[cutoff1:].copy()
cutoff2 = int(0.90 * len(dev_df))
train_df = dev_df.iloc[:cutoff2].copy()
val_df   = dev_df.iloc[cutoff2:].copy()

X_train, y_train, _ = create_sequences(train_df, feature_cols, WINDOW, STRIDE)
X_val,   y_val,   _ = create_sequences(val_df,   feature_cols, WINDOW, STRIDE)

# Test: known + each zero-day family separately (no cross-boundary windows)
X_known, y_known, _ = create_sequences(test_normal, feature_cols, WINDOW, STRIDE)
parts_X, parts_y = [X_known], [y_known]
for fam in zero_day_attacks:
    fam_df = zero_day_df[zero_day_df['Attack'].str.lower() == fam]
    if len(fam_df) >= WINDOW:
        Xf, yf, _ = create_sequences(fam_df, feature_cols, WINDOW, STRIDE)
        if len(Xf) > 0:
            parts_X.append(Xf)
            parts_y.append(yf)
X_test = np.concatenate(parts_X, axis=0)
y_test = np.concatenate(parts_y, axis=0)

X_train, y_train, X_val, y_val, X_test, y_test = scale_and_oversample(
    X_train, y_train, X_test, y_test, X_val, y_val)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("Benign in test:", int((y_test == 0).sum()))
# Test size should be ~64216

f1_control = train_and_eval(X_train, y_train, X_test, y_test,
                            "CONTROL (chronological, hosts may overlap)",
                            X_val, y_val)

del train_df, val_df, test_normal, zero_day_df, normal_df, df_tmp
gc.collect()

CONTROL: chronological split (hosts may overlap)
Train: (316512, 10, 20) | Val: (21162, 10, 20) | Test: (64216, 10, 20)
Benign in test: 15281

Epoch 1/40
1237/1237 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9484 - loss: 0.1417 - val_accuracy: 0.9563 - val_loss: 0.1078
Epoch 2/40
1237/1237 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9707 - loss: 0.0830 - val_accuracy: 0.9719 - val_loss: 0.0748
Epoch 3/40
1237/1237 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9754 - loss: 0.0697 - val_accuracy: 0.9804 - val_loss: 0.0568
Epoch 4/40
1237/1237 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9779 - loss: 0.0628 - val_accuracy: 0.9819 - val_loss: 0.0539
Epoch 5/40
1237/1237 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9796 - loss: 0.0579 - val_accuracy: 0.9844 - val_loss: 0.0477
Epoch 6/40
1237/1237 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9806 - loss: 0.0547 - val_accuracy: 0.9853 - val_loss: 0.0463
Epoch 7/40
1237/1237 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9815 - lo

6927

In [5]:
print("="*70)
print("EXPERIMENT A: host-disjoint, all attack types")
print("="*70)

hosts = df['IPV4_SRC_ADDR'].unique()
train_hosts, test_hosts = train_test_split(hosts, test_size=0.20, random_state=42)

train_df = df[df['IPV4_SRC_ADDR'].isin(train_hosts)].copy()
test_df  = df[df['IPV4_SRC_ADDR'].isin(test_hosts)].copy()

# small val from train hosts (order-preserving)
cutoff = int(0.90 * len(train_df))
val_df   = train_df.iloc[cutoff:].copy()
train_df = train_df.iloc[:cutoff].copy()

X_train, y_train, _ = create_sequences(train_df, feature_cols, WINDOW, STRIDE)
X_val,   y_val,   _ = create_sequences(val_df,   feature_cols, WINDOW, STRIDE)
X_test,  y_test,  _ = create_sequences(test_df,  feature_cols, WINDOW, STRIDE)

X_train, y_train, X_val, y_val, X_test, y_test = scale_and_oversample(
    X_train, y_train, X_test, y_test, X_val, y_val)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)

f1_A = train_and_eval(X_train, y_train, X_test, y_test,
                      "EXPERIMENT A (host-disjoint, all attacks)",
                      X_val, y_val)

del train_df, val_df, test_df
gc.collect()

EXPERIMENT A: host-disjoint, all attack types
Train: (309136, 10, 20) | Val: (22162, 10, 20) | Test: (54216, 10, 20)
Epoch 1/40
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9397 - loss: 0.1608 - val_accuracy: 0.9763 - val_loss: 0.0634
Epoch 2/40
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9658 - loss: 0.0969 - val_accuracy: 0.9836 - val_loss: 0.0468
Epoch 3/40
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9706 - loss: 0.0835 - val_accuracy: 0.9870 - val_loss: 0.0383
Epoch 4/40
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9733 - loss: 0.0759 - val_accuracy: 0.9896 - val_loss: 0.0325
Epoch 5/40
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9753 - loss: 0.0700 - val_accuracy: 0.9905 - val_loss: 0.0298
Epoch 6/40
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9769 - loss: 0.0662 - val_accuracy: 0.9913 - val_loss: 0.0279
Epoch 7/40
1208/1208 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9776 - loss: 0.0629 - val_accuracy:

6937

In [6]:
# ============================================================
# EXPERIMENT C — host-disjoint known + all zero-day
# Test sequences built PER PARTITION (no cross-boundary windows)
# ============================================================
print("="*70)
print("EXPERIMENT C: host-disjoint known + all zero-day in test")
print("="*70)

hosts = df['IPV4_SRC_ADDR'].unique()
train_hosts, test_hosts = train_test_split(hosts, test_size=0.20, random_state=42)

train_mask = (
    df['IPV4_SRC_ADDR'].isin(train_hosts) &
    ~df['Attack'].str.lower().isin(zero_day_attacks)
)
train_full = df.loc[train_mask].copy()

cutoff = int(0.90 * len(train_full))
val_df   = train_full.iloc[cutoff:].copy()
train_df = train_full.iloc[:cutoff].copy()

test_known = df[
    df['IPV4_SRC_ADDR'].isin(test_hosts) &
    ~df['Attack'].str.lower().isin(zero_day_attacks)
].copy()
zero_day_df = df[df['Attack'].str.lower().isin(zero_day_attacks)].copy()

X_train, y_train, _ = create_sequences(train_df, feature_cols, WINDOW, STRIDE)
X_val,   y_val,   _ = create_sequences(val_df,   feature_cols, WINDOW, STRIDE)

# Test: known + each zero-day family separately
X_known, y_known, _ = create_sequences(test_known, feature_cols, WINDOW, STRIDE)
parts_X, parts_y = [X_known], [y_known]
for fam in zero_day_attacks:
    fam_df = zero_day_df[zero_day_df['Attack'].str.lower() == fam]
    if len(fam_df) >= WINDOW:
        Xf, yf, _ = create_sequences(fam_df, feature_cols, WINDOW, STRIDE)
        if len(Xf) > 0:
            parts_X.append(Xf)
            parts_y.append(yf)
X_test = np.concatenate(parts_X, axis=0)
y_test = np.concatenate(parts_y, axis=0)

X_train, y_train, X_val, y_val, X_test, y_test = scale_and_oversample(
    X_train, y_train, X_test, y_test, X_val, y_val)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("Benign in test (before predict):", int((y_test == 0).sum()))

f1_C = train_and_eval(X_train, y_train, X_test, y_test,
                      "EXPERIMENT C (host-disjoint known + all zero-day)",
                      X_val, y_val)

del train_df, val_df, train_full, test_known, zero_day_df
gc.collect()

EXPERIMENT C: host-disjoint known + all zero-day in test
Train: (294644, 10, 20) | Val: (21320, 10, 20) | Test: (62638, 10, 20)
Benign in test (before predict): 422
Epoch 1/40
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9478 - loss: 0.1402 - val_accuracy: 0.9721 - val_loss: 0.0692
Epoch 2/40
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9723 - loss: 0.0793 - val_accuracy: 0.9831 - val_loss: 0.0461
Epoch 3/40
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9763 - loss: 0.0679 - val_accuracy: 0.9861 - val_loss: 0.0380
Epoch 4/40
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9786 - loss: 0.0615 - val_accuracy: 0.9869 - val_loss: 0.0345
Epoch 5/40
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9806 - loss: 0.0562 - val_accuracy: 0.9884 - val_loss: 0.0311
Epoch 6/40
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9813 - loss: 0.0530 - val_accuracy: 0.9901 - val_loss: 0.0285
Epoch 7/40
1151/1151 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step -

6925

In [7]:
print("\n" + "="*70)
print("HOST-AWARE DIAGNOSTICS SUMMARY (primary FB-HESN-64, ω=[0.3,1.0])")
print("="*70)
print(f"CONTROL Macro-F1 : {f1_control:.4f}")
print(f"Exp A Macro-F1   : {f1_A:.4f}")
print(f"Exp C Macro-F1   : {f1_C:.4f}")


HOST-AWARE DIAGNOSTICS SUMMARY (primary FB-HESN-64, ω=[0.3,1.0])
CONTROL Macro-F1 : 0.9471
Exp A Macro-F1   : 0.9573
Exp C Macro-F1   : 0.7443
